Execute the next two cells to open the visualization in the right window and to setup the demo.
Once the demo is setup three text fields with queries that exemplify the three steps of the law of task achieving body motion appear.
You can execute a query with the "Run Query" button and ask for the next solution with the "Next Solution" button until it prints "No more solutions".
Some queries produce visualized outputs in the right window, also the query text can be changed if you like.

In [ ]:
%%bash --bg
rviz -d /home/jovyan/giskard_examples/launch/rvizweb_config/bmp.rviz

In [ ]:
from demo import setup_demo
setup_demo()

In [1]:
from pyswip import Prolog, registerForeign

In [73]:
import os
from ripple_down_rules.datastructures.dataclasses import CaseQuery
from ripple_down_rules.rdr import GeneralRDR
from semantic_world.adapters.urdf import URDFParser
from semantic_world.views import *

from semantic_world.world import World
from semantic_world.views.world_rdr.world_rdr import classify
from semantic_world.connections import RevoluteConnection, PrismaticConnection

dlr_kitchen = os.path.join("dlr_kitchen.urdf")
dlr_kitchen_parser = URDFParser(dlr_kitchen)
world = dlr_kitchen_parser.parse()
world.validate()
found_views = classify(world)['views']
fridge_container_names = [v.body.name.name for v in found_views if isinstance(v, Fridge)]
container_names = [v.body.name.name for v in found_views if isinstance(v, Container)]
cabinet_names = [v.container.body.name.name for v in found_views if isinstance(v, Cabinet)]
drawer_container_names = [v.container.body.name.name for v in found_views if isinstance(v, Drawer)]
handle_names = [v.body.name.name for v in found_views if isinstance(v, Handle)]
door_names = [v.body.name.name for v in found_views if isinstance(v, Door)]
print(f"Fridge container names: {fridge_container_names}")
print(f"Container names: {container_names}")
print(f"Cabinet names: {cabinet_names}")
print(f"Drawer container names: {drawer_container_names}")
print(f"Handle names: {handle_names}")
print(f"Door names: {door_names}")
# For a given container give me the name of the movable connection
fridge_instances = [v for v in found_views if isinstance(v, Fridge)]
fridge_instance_connections = [c for c in world.connections if isinstance(c, RevoluteConnection) and
                               c.parent == fridge_instances[0].body]
fridge_instance_connections_names = [c.dof.name.name for c in fridge_instance_connections]
print(f"Fridge instance connections names: {fridge_instance_connections_names}")
fridge_connection_names = [c.child.name.name for c in world.connections if isinstance(c, RevoluteConnection) and
                           c.parent in [v.body for v in found_views if isinstance(v, Fridge)]]
print(fridge_connection_names)
#Todo drawer connections

Fridge container names: ['fridge']
Container names: ['drawer_04', 'drawer_06', 'drawer_01', 'drawer_03', 'drawer_05', 'drawer_02', 'oven_drawer']
Cabinet names: ['kitchenette']
Drawer container names: ['drawer_01', 'drawer_02', 'drawer_04', 'drawer_06', 'oven_drawer', 'drawer_03', 'drawer_05']
Handle names: ['drawer_05_handle', 'drawer_06_handle', 'drawer_02_handle', 'oven_drawer_handle', 'drawer_03_handle', 'drawer_01_handle', 'drawer_04_handle', 'oven_door_handle', 'fridge_door_handle']
Door names: ['oven_door', 'fridge_door']
Fridge instance connections names: ['fridge_door_joint']
['fridge_door']


In [70]:
def container_articulation(container, handle, joint):
    c = str(container)
    container_instance = [v for v in found_views if (isinstance(v, Fridge) or isinstance(v, Container)) and v.body.name.name == c]
    if len(container_instance) == 1:
        instance_connections = [c.dof.name.name for c in world.connections if (isinstance(c, RevoluteConnection) or isinstance(c, PrismaticConnection)) and
                                (c.parent == container_instance[0].body or c.child == container_instance[0].body)]
        if len(instance_connections) == 0:
            return False
        handle_name = [v.body.name.name for v in found_views if isinstance(v, Handle) and c in v.body.name.name]
        if len(handle_name) == 0:
            return False
        handle.unify(handle_name[0])
        joint.unify(instance_connections[0])
        return True
    return False

def load_container(pl):
    container_list = [v.body.name.name for v in found_views if isinstance(v, Fridge)] + [v.body.name.name for v in found_views if isinstance(v, Container)]
    for container in container_list:
        pl.assertz(f'container({container})')



In [71]:
prolog = Prolog()
registerForeign(container_articulation, arity=3)

load_container(prolog)

In [72]:
res = prolog.query('container(X).')
for result in res:
    print(result)

{'X': 'fridge'}
{'X': 'drawer_04'}
{'X': 'drawer_06'}
{'X': 'drawer_01'}
{'X': 'drawer_03'}
{'X': 'drawer_05'}
{'X': 'drawer_02'}
{'X': 'oven_drawer'}


In [27]:
fridge_instances = [v for v in found_views if isinstance(v, Container)]
fridge_instance_connections = [c for c in world.connections if isinstance(c, PrismaticConnection)]
fridge_instance_connections_names = [c.dof.name.name for c in fridge_instance_connections]
print(fridge_instances)
print(fridge_instance_connections)
print(fridge_instance_connections_names)

[Container(body=Body(name=PrefixedName(name='drawer_04', prefix='dlr_kitchen'))), Container(body=Body(name=PrefixedName(name='drawer_06', prefix='dlr_kitchen'))), Container(body=Body(name=PrefixedName(name='drawer_01', prefix='dlr_kitchen'))), Container(body=Body(name=PrefixedName(name='drawer_03', prefix='dlr_kitchen'))), Container(body=Body(name=PrefixedName(name='drawer_05', prefix='dlr_kitchen'))), Container(body=Body(name=PrefixedName(name='drawer_02', prefix='dlr_kitchen'))), Container(body=Body(name=PrefixedName(name='oven_drawer', prefix='dlr_kitchen')))]
[PrismaticConnection(parent=Body(name=PrefixedName(name='kitchenette', prefix='dlr_kitchen')), child=Body(name=PrefixedName(name='drawer_02', prefix='dlr_kitchen')), origin=SX(@1=1, @2=0, 
[[@1, @2, @2, (-0.148-None/drawer_02_joint_0)], 
 [@2, @1, @2, 0.616], 
 [@2, @2, @1, -0.157], 
 [00, 00, 00, @1]]), active_dofs=[DegreeOfFreedom(name=PrefixedName(name='drawer_02_joint', prefix=None), _lower_limits={<Derivatives.position: 0

In [40]:
fridge_instance_connections[0].child


Body(name=PrefixedName(name='drawer_02', prefix='dlr_kitchen'))

In [65]:
[v.body.name.name for v in found_views if isinstance(v, Handle) and 'drawer_01' in v.body.name.name]

['drawer_01_handle']